In [1]:
import pandas as pd
import numpy as np

# Carregar o dataset. Certifique-se de que o arquivo está no mesmo diretório.
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Exibe as primeiras 5 linhas para inspeção
print("Primeiras 5 linhas do DataFrame:")
print(df.head())

Primeiras 5 linhas do DataFrame:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV Str

In [2]:
# A coluna TotalCharges contém espaços vazios (' ') que devem ser tratados como NaN.
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)

# Converte a coluna para o tipo numérico (float)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])

# Verificação: Exibe os tipos de dados após a conversão para confirmar
print("Tipos de dados após a conversão de TotalCharges:")
print(df.dtypes)

Tipos de dados após a conversão de TotalCharges:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object


In [3]:
# 1. Inspeciona a contagem de valores ausentes (NaN)
print("Contagem de valores ausentes por coluna (após a limpeza):")
print(df.isnull().sum())

# 2. Remove linhas com valores ausentes.
# Como a quantidade é pequena (geralmente menos de 1%), a remoção é segura.
df.dropna(inplace=True)

# 3. Remove a coluna 'customerID'
# Esta coluna é um identificador e não contribui para a previsão do modelo.
df.drop(columns=['customerID'], inplace=True)

# Exibe o formato final do DataFrame e o número de linhas removidas (se houver)
print("\nFormato do DataFrame após limpeza de NaN e remoção de ID:")
print(df.shape)

Contagem de valores ausentes por coluna (após a limpeza):
customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

Formato do DataFrame após limpeza de NaN e remoção de ID:
(7032, 20)


In [4]:
# Exibe a distribuição da variável alvo 'Churn' (sim ou não)
print("\nDistribuição da Variável Alvo 'Churn':")
# 'Yes' (Sim) representa Churn e 'No' (Não) representa retenção.
print(df['Churn'].value_counts(normalize=True))

# Separa a variável alvo (y) das features (X)
X = df.drop('Churn', axis=1)
y = df['Churn']


Distribuição da Variável Alvo 'Churn':
Churn
No     0.734215
Yes    0.265785
Name: proportion, dtype: float64


In [5]:
# 1. Codificação da Variável Alvo 'Churn' ('Yes'/'No') para 1/0 (Label Encoding)
# Embora seja binária, o One-Hot Encoding no passo 2 tratará outras binárias
y_encoded = y.replace({'Yes': 1, 'No': 0})

# 2. Aplicar One-Hot Encoding no conjunto de FEATURES (X)
# Usamos o 'drop_first=True' para evitar multicolinearidade
X_processed = pd.get_dummies(X)

# Exibe o formato final após a codificação para confirmar o aumento de colunas
print(f"Número de Features (Colunas) após a codificação: {X_processed.shape[1]}")
print(X_processed.head(2))

Número de Features (Colunas) após a codificação: 45
   SeniorCitizen  tenure  MonthlyCharges  TotalCharges  gender_Female  \
0              0       1           29.85         29.85           True   
1              0      34           56.95       1889.50          False   

   gender_Male  Partner_No  Partner_Yes  Dependents_No  Dependents_Yes  ...  \
0        False       False         True           True           False  ...   
1         True        True        False           True           False  ...   

   StreamingMovies_Yes  Contract_Month-to-month  Contract_One year  \
0                False                     True              False   
1                False                    False               True   

   Contract_Two year  PaperlessBilling_No  PaperlessBilling_Yes  \
0              False                False                  True   
1              False                 True                 False   

   PaymentMethod_Bank transfer (automatic)  \
0                              

C:\Users\RonaldoRosarioRamos\AppData\Local\Temp\ipykernel_23856\2673555457.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_encoded = y.replace({'Yes': 1, 'No': 0})


In [6]:
from sklearn.model_selection import train_test_split

# Define a proporção da divisão (80% treino, 20% teste)
TEST_SIZE = 0.2
RANDOM_STATE = 42 # Garante que a divisão seja a mesma sempre

# O parâmetro 'stratify=y_encoded' garante que a proporção de Churn (1 e 0)
# seja mantida em ambos os conjuntos (crucial para dados desbalanceados).
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, 
    y_encoded, 
    test_size=TEST_SIZE, 
    random_state=RANDOM_STATE,
    stratify=y_encoded
)

print(f"Tamanho do conjunto de Treinamento: {X_train.shape[0]} linhas")
print(f"Tamanho do conjunto de Teste: {X_test.shape[0]} linhas")

Tamanho do conjunto de Treinamento: 5625 linhas
Tamanho do conjunto de Teste: 1407 linhas


In [7]:
from sklearn.preprocessing import StandardScaler

# Identifica as colunas numéricas (que não são resultado do One-Hot Encoding)
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Inicializa o Scaler
scaler = StandardScaler()

# 1. Treina o Scaler APENAS com os dados de TREINAMENTO (fit_transform)
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

# 2. Aplica o Scaler no conjunto de TESTE usando as estatísticas do TREINAMENTO (transform)
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("Dados numéricos reescalonados com sucesso!")
print(X_train[numeric_cols].head(3))

Dados numéricos reescalonados com sucesso!
        tenure  MonthlyCharges  TotalCharges
1413  1.321816        0.981556      1.659900
7003 -0.267410       -0.971546     -0.562252
3355  1.444064        0.837066      1.756104


In [8]:
from sklearn.linear_model import LogisticRegression

# 1. Inicializa o modelo
# Usamos 'solver' e 'random_state' para garantir a reprodutibilidade
log_model = LogisticRegression(solver='liblinear', random_state=42)

# 2. Treina o modelo com os dados de treinamento (X_train e y_train)
log_model.fit(X_train, y_train)

print("Modelo de Regressão Logística treinado com sucesso!")

Modelo de Regressão Logística treinado com sucesso!


In [9]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

# 1. Faz previsões no conjunto de TESTE
y_pred = log_model.predict(X_test)
y_proba = log_model.predict_proba(X_test)[:, 1] # Probabilidade da classe positiva (Churn=1)

# 2. Exibe a Matriz de Confusão (Ajuda a ver Falsos Positivos e Falsos Negativos)
print("Matriz de Confusão:\n", confusion_matrix(y_test, y_pred))

# 3. Exibe o Relatório de Classificação (Acurácia, Precisão, Recall e F1-Score)
print("\nRelatório de Classificação:\n", classification_report(y_test, y_pred))

# 4. Calcula a métrica ROC AUC (Area Under the Curve), excelente para modelos desbalanceados
roc_auc = roc_auc_score(y_test, y_proba)
print(f"\nROC AUC Score: {roc_auc:.4f}")

Matriz de Confusão:
 [[917 116]
 [160 214]]

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.85      0.89      0.87      1033
           1       0.65      0.57      0.61       374

    accuracy                           0.80      1407
   macro avg       0.75      0.73      0.74      1407
weighted avg       0.80      0.80      0.80      1407


ROC AUC Score: 0.8359
